# MS-ILA — Day-03 QA Report

Mục tiêu chính của notebook là xác nhận **real cached training step**:

`Cached feature → model → loss → backward → optimizer.step`

Đây là **architecture/training infrastructure QA**, không phải AU-PRO\(_{0.05}\)
evaluation hay bằng chứng generalization.


In [ ]:
# ============================================================
# 0. CONFIG — CHỈNH CELL NÀY
# ============================================================
REPO_URL = "https://github.com/haduckien-raccoon/msila.git"
BRANCH = "main"
PROJECT_DIR = "/content/msila"

# Nếu Task 14 chưa được commit lên branch, notebook sẽ yêu cầu upload ZIP.
AUTO_UPLOAD_TASK14_IF_MISSING = True

# -------- Real feature cache (optional / recommended) --------
FEATURE_CACHE_DIR = None
FEATURE_TRAIN_INDEX = None
FEATURE_MASK_ROOT = None
FEATURE_FUSION_DIM = 64

# Khi đã có cache thật, nên đổi thành True để Day-03 PASS bắt buộc real-cache.
REQUIRE_REAL_CACHE = False

# Chạy toàn bộ tests/ có thể lâu hơn; nhóm Day-3 luôn được chạy.
RUN_FULL_REPO_TEST_SUITE = False

SEED = 2026


In [ ]:
# ============================================================
# 1. Clone repository + provenance
# ============================================================
import os, sys, json, shutil, platform, subprocess, zipfile, textwrap
from pathlib import Path

project = Path(PROJECT_DIR)
if project.exists():
    shutil.rmtree(project)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, PROJECT_DIR],
    check=True,
)
os.chdir(PROJECT_DIR)

commit_sha = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    text=True,
).strip()

print("Repo  :", REPO_URL)
print("Branch:", BRANCH)
print("Commit:", commit_sha)


In [ ]:
# ============================================================
# 2. Overlay Task 14 nếu branch chưa chứa file mới
# ============================================================
required_task14 = [
    Path("src/models/cached_training.py"),
    Path("tests/test_day03_cached_train_step.py"),
]

missing = [p for p in required_task14 if not p.is_file()]

if missing and AUTO_UPLOAD_TASK14_IF_MISSING:
    from google.colab import files

    print("Task 14 chưa có trên branch.")
    print("Upload file day3_task14_cached.zip hoặc combined ZIP được bàn giao.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Không nhận được ZIP Task 14.")

    uploaded_name = next(iter(uploaded))
    uploaded_path = Path("/content") / uploaded_name

    extract_root = Path("/content/task14_overlay")
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True)

    with zipfile.ZipFile(uploaded_path, "r") as zf:
        zf.extractall(extract_root)

    candidates = [extract_root, *[p for p in extract_root.rglob("*") if p.is_dir()]]
    overlay_root = None
    for candidate in candidates:
        if (candidate / "src/models/cached_training.py").is_file():
            overlay_root = candidate
            break

    if overlay_root is None:
        raise FileNotFoundError(
            "ZIP không chứa src/models/cached_training.py"
        )

    for rel in [
        Path("src/models/cached_training.py"),
        Path("tests/test_day03_cached_train_step.py"),
    ]:
        src = overlay_root / rel
        dst = project / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print("Overlay:", rel)

missing = [str(p) for p in required_task14 if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Task 14 vẫn thiếu. Commit file lên branch hoặc bật upload overlay. "
        f"Missing: {missing}"
    )

print("Task-14 source presence: PASS")


In [ ]:
# ============================================================
# 3. Dependencies
# ============================================================
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "pytest", "numpy", "pillow", "matplotlib", "safetensors"
    ],
    check=True,
)

import torch
import numpy as np

environment = {
    "repo_url": REPO_URL,
    "branch": BRANCH,
    "commit_sha": commit_sha,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "cuda_version": torch.version.cuda,
    "device": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "cpu"
    ),
}
print(json.dumps(environment, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 4. Helper chạy pytest và giữ stdout/stderr cho report
# ============================================================
def run_pytest(args, env=None):
    command = [sys.executable, "-m", "pytest", "-q", *args]
    print("$", " ".join(command))

    completed = subprocess.run(
        command,
        text=True,
        capture_output=True,
        env=env,
    )
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)

    return {
        "command": command,
        "returncode": completed.returncode,
        "stdout": completed.stdout,
        "stderr": completed.stderr,
        "pass": completed.returncode == 0,
    }


## 5. Mandatory controlled cached train-step gate

Gate này phải thực sự đi qua:

`FeatureCacheWriter → CachedFeatureDataset → Adapter → Alignment → Projection → Fusion → Decoder → BCE+Dice → backward → optimizer.step`


In [ ]:
controlled_cached = run_pytest([
    "tests/test_day03_cached_train_step.py::test_cached_feature_to_optimizer_step_end_to_end",
    "tests/test_day03_cached_train_step.py::test_cached_geometry_inverse_fallback_is_exercised",
])

if not controlled_cached["pass"]:
    raise RuntimeError("Task-14 controlled cached integration FAILED.")

print("CONTROLLED CACHED TRAIN STEP: PASS")


## 6. Day-3 test sweep

Chỉ file thực sự tồn tại mới được chạy. File thiếu được ghi rõ trong report,
không tự động coi là PASS.


In [ ]:
day3_candidates = [
    "tests/test_synthetic_anomaly.py",
    "tests/test_build_overfit16.py",
    "tests/test_anomaly_loss.py",
    "tests/test_optimizer_setup.py",
    "tests/test_overfit16_trainer.py",
    "tests/test_training_visualization.py",
    "tests/test_checkpoint.py",
    "tests/test_resume.py",
    "tests/test_day03_cached_train_step.py",
]

existing_day3 = [p for p in day3_candidates if Path(p).is_file()]
missing_day3 = [p for p in day3_candidates if not Path(p).is_file()]

day3_suite = run_pytest(existing_day3)

print("Existing Day-3 tests:", existing_day3)
print("Missing Day-3 tests :", missing_day3)

if RUN_FULL_REPO_TEST_SUITE:
    full_suite = run_pytest(["tests"])
else:
    full_suite = {
        "pass": None,
        "status": "SKIPPED",
        "reason": "RUN_FULL_REPO_TEST_SUITE=False",
    }


## 7. Real cached one-step acceptance gate

Chỉ chạy khi `FEATURE_CACHE_DIR` và `FEATURE_TRAIN_INDEX` được cấu hình.


In [ ]:
real_cache = {
    "status": "SKIPPED",
    "reason": "FEATURE_CACHE_DIR / FEATURE_TRAIN_INDEX not configured",
}

if FEATURE_CACHE_DIR is not None and FEATURE_TRAIN_INDEX is not None:
    env = os.environ.copy()
    env["FEATURE_CACHE_DIR"] = str(FEATURE_CACHE_DIR)
    env["FEATURE_TRAIN_INDEX"] = str(FEATURE_TRAIN_INDEX)
    env["FEATURE_FUSION_DIM"] = str(FEATURE_FUSION_DIM)

    if FEATURE_MASK_ROOT is not None:
        env["FEATURE_MASK_ROOT"] = str(FEATURE_MASK_ROOT)

    real_pytest = run_pytest(
        [
            "tests/test_day03_cached_train_step.py::"
            "test_external_real_cache_one_training_step",
            "-s",
        ],
        env=env,
    )

    real_cache = {
        "status": "PASS" if real_pytest["pass"] else "FAIL",
        "cache_dir": str(FEATURE_CACHE_DIR),
        "train_index": str(FEATURE_TRAIN_INDEX),
        "mask_root": (
            None if FEATURE_MASK_ROOT is None else str(FEATURE_MASK_ROOT)
        ),
        "fusion_dim": FEATURE_FUSION_DIM,
        "pytest": real_pytest,
    }

    if REQUIRE_REAL_CACHE and not real_pytest["pass"]:
        raise RuntimeError("REAL CACHE train-step gate FAILED.")

print(json.dumps(real_cache, indent=2, ensure_ascii=False))


## 8. Verdict

`REQUIRE_REAL_CACHE=True` nên được dùng khi nhóm đã có cache thật sẵn sàng cho
bàn giao Day-3.


In [ ]:
controlled_pass = bool(controlled_cached["pass"])
day3_pass = bool(day3_suite["pass"])

if REQUIRE_REAL_CACHE:
    real_required_pass = real_cache.get("status") == "PASS"
else:
    real_required_pass = real_cache.get("status") in {"PASS", "SKIPPED"}

overall_pass = controlled_pass and day3_pass and real_required_pass

verdict = {
    "overall_status": "PASS" if overall_pass else "FAIL",
    "controlled_cached_integration": (
        "PASS" if controlled_pass else "FAIL"
    ),
    "day3_tests": "PASS" if day3_pass else "FAIL",
    "missing_day3_tests": missing_day3,
    "real_cache": real_cache.get("status"),
    "real_cache_required": REQUIRE_REAL_CACHE,
    "scientific_scope": (
        "Architecture/training infrastructure QA only; "
        "not generalization, AU-PRO0.05, or baseline superiority."
    ),
}

print(json.dumps(verdict, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 9. Export report
# ============================================================
from datetime import datetime, timezone

report_dir = Path("reports")
report_dir.mkdir(parents=True, exist_ok=True)

report = {
    "schema": "msila.day03.qa.v1",
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "environment": environment,
    "verdict": verdict,
    "controlled_cached": controlled_cached,
    "day3_suite": day3_suite,
    "full_suite": full_suite,
    "real_cache": real_cache,
}

json_path = report_dir / "day03_qa_report.json"
json_path.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

md = f'''# MS-ILA Day-03 QA Report

- Repository: `{REPO_URL}`
- Branch: `{BRANCH}`
- Commit: `{commit_sha}`
- Overall: **{verdict["overall_status"]}**
- Controlled cached train step: **{verdict["controlled_cached_integration"]}**
- Day-3 tests: **{verdict["day3_tests"]}**
- Real cache: **{verdict["real_cache"]}**
- Real cache required: `{REQUIRE_REAL_CACHE}`

## Scientific scope

PASS chỉ chứng minh architecture/training infrastructure contract hoạt động,
đặc biệt cached feature -> loss -> backward -> optimizer update.

PASS không chứng minh generalization, AU-PRO0.05, hoặc superiority so với baseline.
'''

md_path = report_dir / "day03_qa_report.md"
md_path.write_text(textwrap.dedent(md).strip() + "\n", encoding="utf-8")

print("Saved:", json_path.resolve())
print("Saved:", md_path.resolve())


In [ ]:
# ============================================================
# 10. Optional download reports from Colab
# ============================================================
try:
    from google.colab import files
    files.download("reports/day03_qa_report.json")
    files.download("reports/day03_qa_report.md")
except Exception as exc:
    print("Download skipped:", exc)
